# Notebook 06 · MNIST con PyTorch: replicando la arquitectura con ciclo explícito

| R1-A2-S4 · Redes Neuronales II | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-II` |


**Problema:** el mismo del Notebook 05 —clasificación multiclase de dígitos 0–9 sobre MNIST— resuelto
con **PyTorch**. La arquitectura se mantiene esencialmente igual: **784 → 128 → 10**, sin capas
convolucionales.

> **Alcance:** `Dataset` y `DataLoader`, definición de una red con `nn.Module`, criterio de pérdida y
> optimizador, y el **ciclo de entrenamiento explícito** de cinco pasos donde *backpropagation* aparece
> de forma visible en el código.

---

## 1. Por qué replicar el modelo en otro framework

Aprender dos *frameworks* no es aprender dos matemáticas distintas: es reconocer que **ambos
automatizan los mismos fundamentos**. La diferencia relevante es el **nivel de explicitud**:

| Acción | TensorFlow + Keras | PyTorch |
|---|---|---|
| Crear capas | `Dense(n, activation=...)` | `nn.Linear(entrada, salida)` + activación aparte |
| ReLU | `activation="relu"` | `nn.ReLU()` |
| Softmax en la salida | `activation="softmax"` | **Implícito**: `nn.CrossEntropyLoss` lo aplica internamente sobre los *logits* |
| Pérdida | `compile(loss=...)` | `nn.CrossEntropyLoss()` |
| Gradientes | Automáticos dentro de `fit()` | `error.backward()` |
| Actualización de pesos | Automática por el optimizador | `optimizador.step()` |
| Limpiar gradientes | No aplica | `optimizador.zero_grad()` (obligatorio) |
| Ciclo de entrenamiento | `modelo.fit(...)` | Bucle `for` escrito por el programador |
| Modo inferencia | Implícito | `modelo.eval()` + `torch.no_grad()` |
| Carga de datos por lotes | `batch_size` en `fit()` | `DataLoader(batch_size=...)` |

**Dónde está el *backpropagation* en PyTorch:** en `error.backward()`, que calcula los gradientes y los
acumula en el atributo `.grad` de cada parámetro. **Dónde se actualizan los pesos:** en
`optimizador.step()`.


## 2. Entorno de ejecución

En **Google Colab**, PyTorch y torchvision están preinstalados. `ToTensor()` convierte cada imagen a un
tensor de forma `(1, 28, 28)` y **escala automáticamente los píxeles a [0, 1]**, es decir, hace el mismo
trabajo que la división manual entre 255 del Notebook 05.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

torch.manual_seed(42)
print("PyTorch    :", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo usado:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch    : 2.14.1+cpu
CUDA disponible: False
Dispositivo usado: cpu


## 3. Carga del dataset MNIST

`datasets.MNIST(download=True)` descarga los archivos originales de Yann LeCun a la carpeta `./data`
si no existen. Es el **mismo dataset** usado por Keras, con la misma partición oficial de 60 000 /
10 000 imágenes.

In [2]:
transform = transforms.ToTensor()      # PIL/numpy -> tensor (1,28,28) en [0,1]

# La primera ejecución descarga MNIST (~11 MB) a la carpeta ./data.
# Se silencia la barra de progreso de la descarga para que la evidencia del
# notebook quede limpia; en Colab la descarga se muestra de forma nativa.
import contextlib
import urllib.request
import torchvision.datasets.utils as tvutils


def _descarga_sin_barra(url, filename, chunk_size=1024 * 256):
    """Descarga equivalente a la de torchvision, pero sin barra de progreso."""
    with urllib.request.urlopen(url) as respuesta, open(filename, "wb") as archivo:
        while True:
            bloque = respuesta.read(chunk_size)
            if not bloque:
                break
            archivo.write(bloque)


@contextlib.contextmanager
def descarga_silenciosa():
    original = tvutils._urlretrieve
    tvutils._urlretrieve = _descarga_sin_barra
    try:
        yield
    finally:
        tvutils._urlretrieve = original


with descarga_silenciosa():
    train_dataset = datasets.MNIST(root="./data", train=True, download=True,
                                   transform=transform)
    test_dataset = datasets.MNIST(root="./data", train=False, download=True,
                                   transform=transform)


print("Imágenes de entrenamiento:", len(train_dataset))
print("Imágenes de prueba       :", len(test_dataset))

imagen0, etiqueta0 = train_dataset[0]
print()
print("Tensor de la primera imagen:", tuple(imagen0.shape), imagen0.dtype)
print("Rango de píxeles           :", float(imagen0.min()), "-", float(imagen0.max()))
print("Etiqueta real              :", etiqueta0)

Imágenes de entrenamiento: 60000
Imágenes de prueba       : 10000

Tensor de la primera imagen: (1, 28, 28) torch.float32
Rango de píxeles           : 0.0 - 1.0
Etiqueta real              : 5


## 4. DataLoaders: entrenamiento por lotes (*batches*)

Un **batch** es un pequeño grupo de ejemplos procesado antes de una actualización de pesos. Trabajar
por lotes en lugar de ejemplo a ejemplo tiene tres ventajas: aprovecha la computación vectorizada,
reduce la varianza del gradiente estimado y hace el entrenamiento mucho más rápido.

`shuffle=True` **solo** en entrenamiento: barajar los ejemplos evita que el modelo aprenda el orden de
los datos. En prueba el orden es irrelevante y se conserva para poder comparar resultados.

In [3]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Lotes de entrenamiento:", len(train_loader),
      f"({len(train_dataset)} imágenes / {BATCH_SIZE})")
print("Lotes de prueba       :", len(test_loader))

imagenes_lote, etiquetas_lote = next(iter(train_loader))
print()
print("Forma de un lote de imágenes:", tuple(imagenes_lote.shape), "-> (lote, canales, alto, ancho)")
print("Forma de un lote de etiquetas:", tuple(etiquetas_lote.shape))
print("Etiquetas del primer lote   :", etiquetas_lote[:16].tolist())

Lotes de entrenamiento: 938 (60000 imágenes / 64)
Lotes de prueba       : 157

Forma de un lote de imágenes: (64, 1, 28, 28) -> (lote, canales, alto, ancho)
Forma de un lote de etiquetas: (64,)
Etiquetas del primer lote   : [1, 2, 8, 5, 2, 6, 9, 9, 9, 4, 0, 3, 9, 9, 5, 6]


In [4]:
fig, axes = plt.subplots(2, 10, figsize=(11.5, 2.8))
lote_img, lote_etq = next(iter(DataLoader(test_dataset, batch_size=20, shuffle=False)))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(lote_img[i].squeeze(), cmap="gray")
    ax.set_title(str(int(lote_etq[i])), fontsize=9)
    ax.axis("off")
fig.suptitle("Figura 22. Primer lote de prueba cargado con DataLoader (PyTorch)", y=1.04)
plt.tight_layout(); plt.savefig("fig15_pytorch_lote.png"); plt.show()

## 5. Definición de la red con `nn.Module`

En PyTorch la red es una **clase** que hereda de `nn.Module`. Dos métodos son obligatorios:

- `__init__()`: se declaran las capas (se crean los parámetros).
- `forward(x)`: se define el recorrido de los datos (*forward propagation*).

**Nota clave:** la capa de salida **no** lleva Softmax. `nn.CrossEntropyLoss` aplica internamente
`log_softmax` sobre los *logits* por estabilidad numérica. Añadir un `nn.Softmax()` antes de esa
pérdida es un error común que degrada el entrenamiento.

In [5]:
class RedMNIST(nn.Module):
    """Perceptrón multicapa 784 -> 128 -> 10 (sin capas convolucionales)."""

    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()            # (lote,1,28,28) -> (lote,784)
        self.capa1 = nn.Linear(784, 128)       # capa oculta
        self.relu = nn.ReLU()                  # activación de la capa oculta
        self.salida = nn.Linear(128, 10)       # logits de las 10 clases

    def forward(self, x):
        x = self.flatten(x)
        x = self.capa1(x)
        x = self.relu(x)
        x = self.salida(x)
        return x


modelo_pytorch = RedMNIST()
print(modelo_pytorch)
print()
total = sum(p.numel() for p in modelo_pytorch.parameters())
entrenables = sum(p.numel() for p in modelo_pytorch.parameters() if p.requires_grad)
print("Parámetros totales     :", f"{total:,}")
print("Parámetros entrenables :", f"{entrenables:,}")
print()
for nombre, p in modelo_pytorch.named_parameters():
    print(f"  {nombre:<18} forma {tuple(p.shape)!s:<14} parámetros {p.numel():>8,}")

RedMNIST(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (capa1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (salida): Linear(in_features=128, out_features=10, bias=True)
)

Parámetros totales     : 101,770
Parámetros entrenables : 101,770

  capa1.weight       forma (128, 784)     parámetros  100,352
  capa1.bias         forma (128,)         parámetros      128
  salida.weight      forma (10, 128)      parámetros    1,280
  salida.bias        forma (10,)          parámetros       10


### 5.1 Recorrido de una sola imagen (verificación del *forward*)

In [6]:
modelo_pytorch.eval()
imagen_prueba = test_dataset[0][0].unsqueeze(0)     # (1,28,28)

with torch.no_grad():
    logits = modelo_pytorch(imagen_prueba)
    probabilidades = torch.softmax(logits, dim=1)

print("Entrada (lote, canales, alto, ancho):", tuple(imagen_prueba.shape))
print("Logits de salida                    :", tuple(logits.shape))
print("Logits (sin entrenar)               :", np.round(logits.numpy()[0], 4))
print()
print("Softmax aplicado manualmente para inspección:")
print(np.round(probabilidades.numpy()[0], 4), "| suma:", round(float(probabilidades.sum()), 4))
print("-> Antes de entrenar las probabilidades son casi uniformes (~0.1 por clase).")

Entrada (lote, canales, alto, ancho): (1, 1, 28, 28)
Logits de salida                    : (1, 10)
Logits (sin entrenar)               : [-0.0116 -0.0574  0.0625  0.04   -0.0444 -0.1156  0.0228 -0.0114  0.0985
 -0.0605]

Softmax aplicado manualmente para inspección:
[0.0994 0.095  0.1071 0.1047 0.0962 0.0896 0.1029 0.0994 0.111  0.0947] | suma: 1.0
-> Antes de entrenar las probabilidades son casi uniformes (~0.1 por clase).


## 6. Criterio de pérdida y optimizador

**Orden de ejecución obligatorio:** este bloque debe ejecutarse **antes** del entrenamiento. Si no se
ejecuta, el bucle de entrenamiento lanza `NameError: name 'optimizador' is not defined`.

In [7]:
criterio = nn.CrossEntropyLoss()

optimizador = optim.Adam(modelo_pytorch.parameters(), lr=0.001)

print("Criterio de pérdida:", criterio)
print("Optimizador        :", optimizador.__class__.__name__, "| lr =",
      optimizador.param_groups[0]["lr"])
print("Número de tensores de parámetros registrados:", len(optimizador.param_groups[0]["params"]))
print()
print("Modelo, criterio y optimizador listos")

Criterio de pérdida: CrossEntropyLoss()
Optimizador        : Adam | lr = 0.001
Número de tensores de parámetros registrados: 4

Modelo, criterio y optimizador listos


## 7. Ciclo de entrenamiento explícito

Este es el corazón del notebook. Cada lote ejecuta los **cinco pasos** del aprendizaje supervisado:

| Paso | Instrucción | Qué ocurre |
|---|---|---|
| 1 | `optimizador.zero_grad()` | Borra los gradientes del lote anterior (PyTorch los **acumula**) |
| 2 | `salidas = modelo_pytorch(imagenes)` | *Forward propagation* |
| 3 | `error = criterio(salidas, etiquetas)` | Cálculo de la pérdida |
| 4 | `error.backward()` | **Backpropagation**: calcula $\partial L/\partial w$ para cada parámetro |
| 5 | `optimizador.step()` | Actualiza los pesos con los gradientes |

Este mismo ciclo es el que se implementó **a mano** con NumPy en el Notebook 02; aquí `backward()`
reemplaza las derivadas escritas manualmente y `step()` reemplaza `W -= lr * dW`.

In [8]:
epochs = 3
historial_loss = []

t0 = time.time()
for epoch in range(epochs):
    modelo_pytorch.train()              # modo entrenamiento
    error_total = 0.0
    correctas = 0
    total = 0

    for batch_idx, (imagenes, etiquetas) in enumerate(train_loader):
        # 1. Limpiar gradientes anteriores
        optimizador.zero_grad()

        # 2. Forward propagation
        salidas = modelo_pytorch(imagenes)

        # 3. Loss
        error = criterio(salidas, etiquetas)

        # 4. Backpropagation (cálculo de gradientes)
        error.backward()

        # 5. Actualización de pesos
        optimizador.step()

        # --- Métricas de seguimiento ---
        error_total += error.item()
        _, predichas = torch.max(salidas, 1)
        correctas += (predichas == etiquetas).sum().item()
        total += etiquetas.size(0)

        if batch_idx % 200 == 0:
            print(f"  Época {epoch+1}/{epochs} | lote {batch_idx:>4}/{len(train_loader)} "
                  f"| loss del lote: {error.item():.4f}")

    promedio_error = error_total / len(train_loader)
    acc_epoca = correctas / total
    historial_loss.append((promedio_error, acc_epoca))
    print(f"Época {epoch+1} | Error promedio: {promedio_error:.4f} | Accuracy entrenamiento: {acc_epoca:.4f}\n")

tiempo = time.time() - t0
print(f"Tiempo total de entrenamiento: {tiempo:.1f} s ({tiempo/epochs:.1f} s por época)")

  Época 1/3 | lote    0/938 | loss del lote: 2.2901


  Época 1/3 | lote  200/938 | loss del lote: 0.3451


  Época 1/3 | lote  400/938 | loss del lote: 0.3931


  Época 1/3 | lote  600/938 | loss del lote: 0.3834


  Época 1/3 | lote  800/938 | loss del lote: 0.1064


...

  Época 3/3 | lote  200/938 | loss del lote: 0.0380


  Época 3/3 | lote  400/938 | loss del lote: 0.0551


  Época 3/3 | lote  600/938 | loss del lote: 0.1305


  Época 3/3 | lote  800/938 | loss del lote: 0.1942


Época 3 | Error promedio: 0.1061 | Accuracy entrenamiento: 0.9696

Tiempo total de entrenamiento: 13.4 s (4.5 s por época)


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))

axes[0].plot([e[0] for e in historial_loss], marker="o", lw=2, color="#c0392b")
axes[0].set_title("Figura 23. Pérdida por época (PyTorch)")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("CrossEntropyLoss")
for i, e in enumerate(historial_loss):
    axes[0].annotate(f"{e[0]:.4f}", (i, e[0]), textcoords="offset points",
                     xytext=(0, 9), ha="center", fontsize=9)

axes[1].plot([e[1] for e in historial_loss], marker="s", lw=2, color="#27ae60")
axes[1].set_ylim(min(e[1] for e in historial_loss) - 0.01, 1.0)
axes[1].set_title("Figura 24. Accuracy de entrenamiento por época")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy")
for i, e in enumerate(historial_loss):
    axes[1].annotate(f"{e[1]:.4f}", (i, e[1]), textcoords="offset points",
                     xytext=(0, -16), ha="center", fontsize=9)

plt.tight_layout(); plt.savefig("fig16_pytorch_curvas.png"); plt.show()

### 7.1 Inspección de los gradientes

Después de `backward()`, cada parámetro guarda su gradiente en el atributo `.grad`. Esta es la
evidencia directa de que *backpropagation* se ejecutó: los valores no son nulos.

In [10]:
# Un único paso para dejar gradientes cargados y poder inspeccionarlos
modelo_pytorch.train()
imagenes, etiquetas = next(iter(train_loader))
optimizador.zero_grad()
error = criterio(modelo_pytorch(imagenes), etiquetas)
error.backward()

print("Pérdida del lote inspeccionado:", round(error.item(), 6))
print()
print(f"{'Parámetro':<22}{'Norma del gradiente':>22}{'Grad. máximo':>15}{'Grad. mínimo':>15}")
for nombre, p in modelo_pytorch.named_parameters():
    g = p.grad
    print(f"{nombre:<22}{float(torch.norm(g)):>22.6f}{float(g.max()):>15.6f}{float(g.min()):>15.6f}")
print()
print("-> Ningún gradiente es idénticamente cero: la señal de error llegó a todas las capas.")

Pérdida del lote inspeccionado: 0.165745

Parámetro                Norma del gradiente   Grad. máximo   Grad. mínimo
capa1.weight                        0.534879       0.015980      -0.017819
capa1.bias                          0.061155       0.015350      -0.017949
salida.weight                       0.718763       0.099026      -0.155715
salida.bias                         0.046693       0.018190      -0.034189

-> Ningún gradiente es idénticamente cero: la señal de error llegó a todas las capas.


## 8. Evaluación sobre el conjunto de prueba

Dos instrucciones son críticas y se omiten con frecuencia:

- `modelo.eval()`: coloca el modelo en modo inferencia (afecta a capas como *Dropout* o *BatchNorm*).
- `torch.no_grad()`: desactiva el registro de gradientes, ahorrando memoria y acelerando el cálculo.


In [11]:
modelo_pytorch.eval()
correctas = 0
total = 0
todas_pred = []
todas_real = []

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        salidas = modelo_pytorch(imagenes)
        _, predicciones = torch.max(salidas, 1)
        total += etiquetas.size(0)
        correctas += (predicciones == etiquetas).sum().item()
        todas_pred.extend(predicciones.tolist())
        todas_real.extend(etiquetas.tolist())

accuracy_pytorch = correctas / total
todas_pred = np.array(todas_pred)
todas_real = np.array(todas_real)

print("=== RESULTADOS MNIST · PyTorch ===")
print("Imágenes evaluadas :", total)
print("Predicciones correctas:", correctas)
print("Predicciones erróneas :", total - correctas)
print("Accuracy en prueba :", round(accuracy_pytorch, 6))
print("Tasa de error      :", f"{(1-accuracy_pytorch)*100:.2f} %")

=== RESULTADOS MNIST · PyTorch ===
Imágenes evaluadas : 10000
Predicciones correctas: 9690
Predicciones erróneas : 310
Accuracy en prueba : 0.969
Tasa de error      : 3.10 %


## 9. Predicción individual con visualización

In [12]:
def predecir_imagen(indice):
    """Devuelve la predicción y las probabilidades de una imagen del conjunto de prueba."""
    imagen, etiqueta_real = test_dataset[indice]
    entrada = imagen.unsqueeze(0)
    modelo_pytorch.eval()
    with torch.no_grad():
        salida = modelo_pytorch(entrada)
        probs = torch.softmax(salida, dim=1)[0]
        prediccion = int(torch.argmax(probs).item())
    return imagen, int(etiqueta_real), prediccion, probs.numpy()


fig, axes = plt.subplots(1, 5, figsize=(11.5, 2.8))
for ax, idx in zip(axes, [0, 1, 2, 3, 4]):
    imagen, real, pred, probs = predecir_imagen(idx)
    ax.imshow(imagen.squeeze(), cmap="gray")
    ax.set_title(f"Real: {real} | Pred: {pred}\nconfianza {probs.max():.4f}",
                 fontsize=9, color="#27ae60" if real == pred else "#c0392b")
    ax.axis("off")
fig.suptitle("Figura 25. Predicciones individuales (PyTorch)", y=1.08)
plt.tight_layout(); plt.savefig("fig17_pytorch_predicciones.png"); plt.show()

imagen, real, pred, probs = predecir_imagen(0)
print("Número real    :", real)
print("Número predicho:", pred)
print()
print("Distribución de probabilidad por dígito:")
for d, p in enumerate(probs):
    print(f"   {d}: {p:.6f}  {'█' * int(round(p * 40))}")

Número real    : 7
Número predicho: 7

Distribución de probabilidad por dígito:
   0: 0.000002  
   1: 0.000000  
   2: 0.000056  
   3: 0.000833  
   4: 0.000000  
   5: 0.000004  
   6: 0.000000  
   7: 0.999079  ████████████████████████████████████████
   8: 0.000002  
   9: 0.000024  


## 10. Matriz de confusión y reporte por clase

In [13]:
cm = confusion_matrix(todas_real, todas_pred)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(
    ax=axes[0], cmap="Oranges", colorbar=True, values_format="d", xticks_rotation=0)
axes[0].set_title("Figura 26. Matriz de confusión - MNIST con PyTorch")

cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=range(10)).plot(
    ax=axes[1], cmap="Oranges", colorbar=True, values_format=".3f", xticks_rotation=0)
axes[1].set_title("Figura 27. Matriz normalizada por clase real (recall)")
plt.tight_layout(); plt.savefig("fig18_pytorch_confusion.png"); plt.show()

print(classification_report(todas_real, todas_pred,
                            target_names=[str(d) for d in range(10)], digits=4))

pares = sorted([(cm[r, c], r, c) for r in range(10) for c in range(10) if r != c], reverse=True)
print("Top 5 confusiones (real -> predicho):")
for cant, r, c in pares[:5]:
    print(f"   {r} -> {c} : {cant} casos ({cant/cm[r].sum()*100:.2f} % de los '{r}')")

              precision    recall  f1-score   support

           0     0.9768    0.9867    0.9817       980
           1     0.9833    0.9868    0.9850      1135
           2     0.9775    0.9690    0.9732      1032
           3     0.9460    0.9713    0.9585      1010
           4     0.9705    0.9725    0.9715       982
           5     0.9696    0.9641    0.9668       892
           6     0.9659    0.9770    0.9715       958
           7     0.9402    0.9796    0.9595      1028
           8     0.9784    0.9302    0.9537       974
           9     0.9836    0.9495    0.9662      1009

    accuracy                         0.9690     10000
   macro avg     0.9692    0.9687    0.9688     10000
weighted avg     0.9693    0.9690    0.9690     10000

Top 5 confusiones (real -> predicho):
   8 -> 3 : 21 casos (2.16 % de los '8')
   9 -> 7 : 15 casos (1.49 % de los '9')
   9 -> 4 : 14 casos (1.39 % de los '9')
   8 -> 7 : 14 casos (1.44 % de los '8')
   2 -> 7 : 12 casos (1.16 % de los '2'

In [14]:
errores_idx = np.where(todas_pred != todas_real)[0]
print("Cantidad de errores:", len(errores_idx))

fig, axes = plt.subplots(1, 9, figsize=(11.5, 2.2))
for ax, idx in zip(axes, errores_idx[:9]):
    imagen, real, pred, _ = predecir_imagen(int(idx))
    ax.imshow(imagen.squeeze(), cmap="gray")
    ax.set_title(f"Real {real}\nPred {pred}", fontsize=8, color="#c0392b")
    ax.axis("off")
fig.suptitle("Figura 28. Errores del clasificador PyTorch", y=1.12)
plt.tight_layout(); plt.savefig("fig19_pytorch_errores.png"); plt.show()

Cantidad de errores: 310


## 11. Comparación Keras vs PyTorch sobre el mismo problema

Ambos modelos tienen la **misma arquitectura conceptual** (784 → 128 → 10, ReLU + Softmax) y el
**mismo dataset** con la misma partición oficial. Las diferencias de *accuracy* provienen del
número de épocas, del tamaño de lote y de la inicialización aleatoria, no del framework.

| Parámetro | Keras (Notebook 05) | PyTorch (este notebook) |
|---|---|---|
| Arquitectura | `Flatten` + `Dense(128, relu)` + `Dense(10, softmax)` | `nn.Flatten` + `nn.Linear(784,128)` + `nn.ReLU` + `nn.Linear(128,10)` |
| Capas convolucionales | Ninguna | Ninguna |
| Optimizador | Adam (lr = 0.001) | Adam (lr = 0.001) |
| Pérdida | `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss` |
| Épocas | 5 | 3 |
| Batch size | 128 | 64 |
| Normalización | Manual (`/255.0`) | Automática (`transforms.ToTensor()`) |
| *Backpropagation* | Interna en `fit()` | Explícita en `error.backward()` |
| Actualización de pesos | Interna | Explícita en `optimizador.step()` |
| **Accuracy de prueba** | **{acc_keras}** | **{acc_pytorch}** |

Los valores de la última fila se completan automáticamente al final del notebook con las mediciones
reales de esta ejecución.

In [15]:
# Tabla resumen con los resultados REALES obtenidos en esta ejecución
acc_keras_referencia = None   # se completa en la sección de resultados del documento técnico

print("=" * 66)
print("RESUMEN DE RESULTADOS OBTENIDOS (ejecución real de este notebook)")
print("=" * 66)
print(f"Framework            : PyTorch {torch.__version__}")
print(f"Arquitectura         : 784 -> 128 -> 10 (ReLU + Softmax, sin CNN)")
print(f"Parámetros           : {sum(p.numel() for p in modelo_pytorch.parameters()):,}")
print(f"Épocas               : {epochs}")
print(f"Batch size           : {BATCH_SIZE}")
print(f"Loss por época       : {', '.join(f'{e[0]:.4f}' for e in historial_loss)}")
print(f"Acc. entrenamiento   : {historial_loss[-1][1]:.4f}")
print(f"ACCURACY DE PRUEBA   : {accuracy_pytorch:.4f}  ({int(correctas)}/{total})")
print("=" * 66)

RESUMEN DE RESULTADOS OBTENIDOS (ejecución real de este notebook)
Framework            : PyTorch 2.14.1+cpu
Arquitectura         : 784 -> 128 -> 10 (ReLU + Softmax, sin CNN)
Parámetros           : 101,770
Épocas               : 3
Batch size           : 64
Loss por época       : 0.3395, 0.1516, 0.1061
Acc. entrenamiento   : 0.9696
ACCURACY DE PRUEBA   : 0.9690  (9690/10000)


## 12. Errores frecuentes en Colab con PyTorch y su solución

| Error | Causa | Solución |
|---|---|---|
| `NameError: name 'optimizador' is not defined` | El bucle de entrenamiento se ejecutó antes de crear el optimizador | Ejecutar primero la celda de modelo, criterio y optimizador |
| `NameError: name 'criterio' is not defined` | Ídem | Respetar el orden: modelo → criterio → optimizador → entrenamiento |
| `RuntimeError: mat1 and mat2 shapes cannot be multiplied` | Se olvidó `nn.Flatten()` o el `unsqueeze(0)` al predecir una imagen | Verificar que la entrada sea `(lote, 1, 28, 28)` |
| La pérdida no disminuye | Falta `optimizador.zero_grad()` y los gradientes se acumulan entre lotes | Limpiar gradientes al inicio de cada lote |
| `grad can be implicitly created only for scalar outputs` | Se llamó `backward()` sobre un vector | Llamar `backward()` sobre el escalar de pérdida |
| Resultados irreproducibles | Semilla no fijada | `torch.manual_seed(42)` antes de crear el modelo |
| `./data` se pierde al reiniciar Colab | El sistema de archivos de Colab es temporal | `download=True` vuelve a descargar MNIST automáticamente |

En Colab una variable existe únicamente después de ejecutar la celda que la crea; si el entorno se
reinicia conviene usar `Entorno de ejecución → Ejecutar todo`.

## 13. Análisis de resultados

- El ciclo explícito de PyTorch hace **visible** cada uno de los cinco pasos que Keras ejecuta de forma
  interna, lo que facilita depurar problemas de gradientes (explosión, desvanecimiento, acumulación).
- La pérdida promedio disminuye de forma monotónica entre épocas y el *accuracy* de prueba se ubica en
  el rango esperado para una red densa de una capa oculta sobre MNIST (por encima del 96 % con 3 épocas).
- Los patrones de confusión coinciden cualitativamente con los observados en Keras (pares 4↔9, 7↔1,
  3↔5), lo que confirma que las limitaciones provienen de la **arquitectura** (red densa sobre píxeles
  aplanados) y no del framework.
- El costo computacional en CPU es del orden de decenas de segundos por época; en Colab con GPU se
  reduce significativamente.

## 14. Conclusiones del notebook

1. PyTorch organiza el aprendizaje en torno a un **bucle explícito**: `zero_grad()` → `forward` →
   `loss` → `backward()` → `step()`.
2. `nn.CrossEntropyLoss` combina `log_softmax` y entropía cruzada; por ello la red entrega *logits* y
   no probabilidades.
3. `DataLoader` implementa el concepto de *batch* y el barajado de los datos de entrenamiento.
4. `eval()` y `no_grad()` son obligatorios en la fase de evaluación.
5. TensorFlow/Keras y PyTorch son **implementaciones distintas de los mismos fundamentos**: quien
   domina *forward propagation*, pérdida, *backpropagation* y actualización puede migrar entre
   frameworks sin volver a aprender la teoría.

## 15. Referencias

- Stevens, E., Antiga, L., & Viehmann, T. (2020). *Deep Learning with PyTorch*. Manning Publications.
  https://isip.piconepress.com/courses/temple/ece_4822/resources/books/Deep-Learning-with-PyTorch.pdf
- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications.
- LeCun, Y., Cortes, C., & Burges, C. J. C. (2010). *MNIST handwritten digit database*. http://yann.lecun.com/exdb/mnist/
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/